In [2]:
import pdfplumber
import pandas as pd
import numpy as np
import re
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# [환율 설정] 100엔 = 약 900원 (1엔 = 9.0원)
EXCHANGE_RATE = 9.0

# 💡 현재 파이썬 파일이 실행되는 '같은 폴더'를 기준 경로로 명시
BASE_DIR = os.getcwd()

# ==========================================
# 1. 국내 상품: 보장명 실제 추출 및 기초 데이터 스캔 (PDF 파서)
# ==========================================
def extract_base_data_from_pdf(file_path, plan_name, base_age=40, gender=1):
    def clean_money(text):
        text = str(text).replace(',', '').replace(' ', '').replace('\n', '')
        if not text or text in ['None', 'nan', '-']: return 0
        if '/' in text: text = text.split('/')[0] 
        match = re.search(r'(\d+)(억|천만|백만|만)', text)
        if match:
            num = int(match.group(1)); unit = match.group(2)
            if unit == '억': return num * 100000000
            elif unit == '천만': return num * 10000000
            elif unit == '백만': return num * 1000000
            elif unit == '만': return num * 10000
        numbers = re.findall(r'\d+', text)
        if numbers:
            base_num = int(numbers[0])
            if '만' in text and base_num < 10000: return base_num * 10000
            return base_num
        return 0

    if not os.path.exists(file_path):
        return None

    gender_text = '남성' if gender == 1 else '여성'
    ml_row = {'상품플랜': plan_name, '기준나이': base_age, '성별': gender_text}
    
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    if not table or len(table) < 2: continue
                    df = pd.DataFrame(table).fillna('')
                    
                    cov_scores = [0] * len(df.columns)
                    amt_scores = [0] * len(df.columns)
                    for c_idx in range(len(df.columns)):
                        for r_idx in range(len(df)):
                            cell = str(df.iloc[r_idx, c_idx]).replace(' ', '').replace('\n', '')
                            if not cell: continue
                            if r_idx < 4:
                                if any(k in cell for k in ['담보명', '보장명', '보장내용', '특약', '항목']): cov_scores[c_idx] += 100
                                if any(k in cell for k in ['가입금액', '지급금액', '보상한도', '금액', '한도액']): amt_scores[c_idx] += 100
                            m = clean_money(cell)
                            if m >= 10000: amt_scores[c_idx] += 2 

                    cov_col = cov_scores.index(max(cov_scores)) if max(cov_scores) > 0 else -1
                    amt_col = amt_scores.index(max(amt_scores)) if max(amt_scores) > 0 else -1

                    if cov_col != -1 and amt_col != -1 and cov_col != amt_col:
                        for r_idx in range(len(df)):
                            raw_cov = str(df.iloc[r_idx, cov_col]).replace('\n', ' ').strip()
                            money = clean_money(df.iloc[r_idx, amt_col])
                            if money > 0 and 2 <= len(raw_cov) <= 40:
                                clean_name = raw_cov.replace(' ', '')
                                if '보험료' not in clean_name and clean_name not in ml_row:
                                    ml_row[clean_name] = money

                    for r_idx in range(len(df)):
                        row_str = "".join([str(x) for x in df.iloc[r_idx]]).replace(' ', '')
                        if any(k in row_str for k in ['보험료', '합계', '남자', '여자']):
                            for c_idx in range(len(df.columns)):
                                m = clean_money(df.iloc[r_idx, c_idx])
                                if 1000 <= m <= 5000000 and '기준보험료' not in ml_row:
                                    ml_row['기준보험료'] = m
        
        # 비정형 표 예외 처리 (Fallback)
        if '기준보험료' not in ml_row or ml_row['기준보험료'] == 0:
            if 'KB_국내' in plan_name: ml_row['기준보험료'] = 7000 
            elif '현대_유학생' in plan_name: ml_row['기준보험료'] = 15500 if gender == 1 else 15500
            elif 'DB_국내여행' in plan_name: ml_row['기준보험료'] = 15620 if gender == 1 else 18540
            
        return ml_row
    except: return None

# ==========================================
# 2. 해외 상품: 일본 CSV 데이터 스캔 (환율 적용 및 10대 기준)
# ==========================================
def extract_base_data_from_csv(file_path, prefix="일본_"):
    def clean_jp_money(text):
        text = str(text).replace(',', '').replace(' ', '').replace('\n', '').replace('엔', '')
        if not text or text in ['None', 'nan', '-']: return 0
        multiplier = 1
        if '억' in text: multiplier = 100000000
        elif '만' in text: multiplier = 10000
        numbers = re.findall(r'\d+', text)
        if numbers: return int(numbers[0]) * multiplier
        return 0

    if not os.path.exists(file_path):
        return []

    try: df = pd.read_csv(file_path, encoding='utf-8-sig')
    except: df = pd.read_csv(file_path, encoding='cp949')
        
    extracted_rows = []
    col_names = df.columns.tolist()
    
    company_col = '보험사' if '보험사' in col_names else col_names[0]
    premium_col = '보험료' if '보험료' in col_names else col_names[1]
    
    for _, row in df.iterrows():
        comp_name = str(row[company_col]).strip()
        # 원화 환산 적용
        base_prem = int(clean_jp_money(row[premium_col]) * EXCHANGE_RATE)
        if base_prem == 0: continue
            
        # 남녀 모두 추출, 10대를 기준으로 고정
        for gender in [1, 0]:
            gender_text = '남성' if gender == 1 else '여성'
            # 모든 상품을 분석 기준에 맞춰 1개월 명시
            plan_suffix = "(1개월_월환산)" 
            
            ml_row = {
                '상품플랜': f"{prefix}{comp_name}{plan_suffix}",
                '기준나이': 10, 
                '성별': gender_text,
                '기준보험료': base_prem,
                '국가': '일본(환산_KRW)' 
            }
            
            for col in col_names:
                if col not in [company_col, premium_col, '순위', '월 기준', 'Unnamed: 11']: 
                    val = int(clean_jp_money(row[col]) * EXCHANGE_RATE)
                    if val > 0:
                        clean_col = col.replace(' ', '').replace('\n', '')
                        ml_row[clean_col] = val
                        
            extracted_rows.append(ml_row)
            
    return extracted_rows

# ==========================================
# 3. 통합 계리적 시뮬레이션 엔진 (10~70대 및 월 환산 정규화)
# ==========================================
def calculate_projected_premium(plan_name, base_p, target_age, gender, is_4th_gen_convert=False):
    if base_p == 0: return 0
    diff = target_age - 40
    projected_p = base_p
    
    # [1] 기존 계리적 위험률 계산
    if 'KB_국내' in plan_name:
        projected_p = base_p * ((1 + 0.0112) ** diff)
    elif '삼성_국내여행' in plan_name:
        if target_age >= 40: projected_p = base_p * (1.0965 ** (diff/10))
        else: projected_p = base_p / (1.0965 ** (-diff/10) if gender == '남성' else 1.467 ** (-diff/10))
    elif 'DB_국내' in plan_name:
        rate = 0.0064 if gender == '남성' else 0.056
        projected_p = base_p * ((1 + rate) ** diff)
        if is_4th_gen_convert and '실손' in plan_name: projected_p = projected_p * 0.5
    elif '현대_유학생' in plan_name:
        projected_p = base_p * (1.25 ** (diff/10))
        if gender == '여성' and target_age >= 40: projected_p = projected_p * 1.15
    elif '삼성_글로벌' in plan_name:
        rate = 0.012 if gender == '남성' else 0.022
        projected_p = base_p * ((1 + rate) ** diff)
    elif '일본' in plan_name:
        projected_p = base_p # 일본 상품 고정가

    # [2] 연납 상품 정규화 (1개월 월납 기준으로 변환)
    # 현대 외국인유학생보험과 삼성 글로벌실손보험은 연납(1년) 기준이므로 12로 나눔
    if '유학생' in plan_name or '글로벌' in plan_name:
        projected_p = projected_p / 12.0

    return int(round(projected_p, -1)) # 10원 단위 반올림

# ==========================================
# 4. 메인 실행 및 데이터 병합
# ==========================================
# 명시적인 폴더 경로 결합
pdf_files = [
    {"path": os.path.join(BASE_DIR, "삼성화재_국내여행보험_상품요약서.pdf"), "plan": "삼성_국내여행"},
    {"path": os.path.join(BASE_DIR, "삼성화재_글로벌케어_실손의료보험_상품요약서.pdf"), "plan": "삼성_글로벌실손"},
    {"path": os.path.join(BASE_DIR, "현대해상_외국인유학생보험_상품요약서.pdf"), "plan": "현대_유학생"},
    {"path": os.path.join(BASE_DIR, "DB손해보험_프로미다이렉트_국내여행보험_상품요약서.pdf"), "plan": "DB_국내여행"},
    {"path": os.path.join(BASE_DIR, "DB손해보험_프로미다이렉트_국내여행실손의료비보험_상품요약서.pdf"), "plan": "DB_국내실손"},
    {"path": os.path.join(BASE_DIR, "KB손해보험_국내여행보험_상품요약서.pdf"), "plan": "KB_국내여행"}
]

csv_files = [
    {"path": os.path.join(BASE_DIR, "일본 카카쿠닷컴 보험료.csv")}
]

all_base_data = []
print(f" 작업 폴더: {BASE_DIR}")
print(" [국내 6종 PDF] & [해외 단일 CSV] 스캔 및 1개월 정규화.")

# 1. 국내 데이터 추출
for gen in [1, 0]:
    for f in pdf_files:
        # 연납 상품은 이름에 (월환산)을 명시적으로 붙여줌
        display_plan_name = f['plan'] + "(1개월_월환산)" if ('유학생' in f['plan'] or '글로벌' in f['plan']) else f['plan']
        data = extract_base_data_from_pdf(f['path'], display_plan_name, gender=gen)
        
        if data: 
            data['국가'] = '한국(KRW)'
            # 함수 내부 로직 작동을 위해 원본 이름 보존
            data['original_plan_name'] = f['plan'] 
            all_base_data.append(data)

# 2. 일본 CSV 데이터 추출
for csv_info in csv_files:
    jp_data_list = extract_base_data_from_csv(csv_info['path'])
    for jp_data in jp_data_list:
        jp_data['original_plan_name'] = jp_data['상품플랜'] 
    all_base_data.extend(jp_data_list)

# 3. 전 연령(10~70대) 시뮬레이션 전개
all_simulated_rows = []
for base_data in all_base_data:
    orig_plan = base_data.pop('original_plan_name', '') # 계산용 원본 이름 빼내기
    
    for age in [10, 20, 30, 40, 50, 60, 70]:
        new_row = base_data.copy()
        new_row['가입나이'] = age
        
        # 보험료 예측 (계산은 원본 플랜명으로 돌려서 /12 로직을 태움)
        new_row['최종보험료'] = calculate_projected_premium(
            orig_plan, 
            base_data.get('기준보험료', 0), 
            age, 
            base_data['성별']
        )
        
        if 'DB_국내실손' in orig_plan:
            new_row['4세대전환_특별할인가'] = calculate_projected_premium(
                orig_plan, base_data.get('기준보험료', 0), age, base_data['성별'], is_4th_gen_convert=True)
        else:
            new_row['4세대전환_특별할인가'] = np.nan
            
        all_simulated_rows.append(new_row)

# 데이터프레임 병합 및 정제
master_df = pd.DataFrame(all_simulated_rows).drop(columns=['기준나이', '기준보험료'], errors='ignore').fillna(np.nan)

meta_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료', '4세대전환_특별할인가']
feature_cols = [c for c in master_df.columns if c not in meta_cols]
master_df = master_df[meta_cols + feature_cols]

# CSV 저장
save_name = os.path.join(BASE_DIR, "통합_보험데이터_최종_1개월_월환산완료.csv")
master_df.to_csv(save_name, index=False, encoding='utf-8-sig')

print(f"\n 전 상품 1개월(월납) 정규화 처리 및 데이터 통합 완료!")
print(f"총 데이터: {len(master_df)}건 / 파싱된 보장항목 수: {len(feature_cols)}개")
print(f" 저장 완료: {save_name}\n")


 작업 폴더: C:\Users\hahas
 [국내 6종 PDF] & [해외 단일 CSV] 스캔 및 1개월 정규화.

 전 상품 1개월(월납) 정규화 처리 및 데이터 통합 완료!
총 데이터: 266건 / 파싱된 보장항목 수: 22개
 저장 완료: C:\Users\hahas\통합_보험데이터_최종_1개월_월환산완료.csv

